In [ ]:
!unzip "/content/NEATM-master.zip"

Archive:  /content/NEATM-master.zip
abfd38ae2591df52fb20627ebe5d96ecf2c56b88
   creating: NEATM-master/
  inflating: NEATM-master/.gitignore  
  inflating: NEATM-master/LICENSE    
  inflating: NEATM-master/README.md  
   creating: NEATM-master/neatm/
  inflating: NEATM-master/neatm/__init__.py  
   creating: NEATM-master/neatm/c++/
  inflating: NEATM-master/neatm/c++/constants.h  
  inflating: NEATM-master/neatm/c++/integral.cpp  
  inflating: NEATM-master/neatm/c++/integral.h  
  inflating: NEATM-master/neatm/c++/jansky.h  
  inflating: NEATM-master/neatm/c++/main.cpp  
  inflating: NEATM-master/neatm/c++/makefile  
  inflating: NEATM-master/neatm/c++/neatm.cpp  
  inflating: NEATM-master/neatm/c++/neatm.h  
  inflating: NEATM-master/neatm/c++/neatm_input_file.cpp  
  inflating: NEATM-master/neatm/c++/neatm_input_file.h  
  inflating: NEATM-master/neatm/neatm.py  
  inflating: NEATM-master/neatm/reflected.py  
  inflating: NEATM-master/setup.py   


In [ ]:
cd "/content/NEATM-master"

/content/NEATM-master


In [ ]:
cd neatm/c++

/content/NEATM-master/neatm/c++


In [ ]:
!apt-get install make

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
make is already the newest version (4.3-4.1build1).
0 upgraded, 0 newly installed, 0 to remove and 8 not upgraded.


In [ ]:
cd /content/NEATM-master/neatm/c++

/content/NEATM-master/neatm/c++


In [ ]:
!make

g++ -c -O3 main.cpp -o main.o
g++ -c -O3 neatm.cpp -o neatm.o
g++ -c -O3 neatm_input_file.cpp -o neatm_input_file.o
g++ -c -O3 integral.cpp -o integral.o
g++ -O3 -o neatm main.o neatm.o neatm_input_file.o integral.o 


In [ ]:
!echo $PATH


/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/content/NEATM-master/neatm/c++/neatm


In [ ]:
!sudo mv /content/NEATM-master/neatm/c++/neatm /usr/local/bin/


In [ ]:
!neatm

Wrong number of command line arguments!
Please provide four or eight arguments:
Four arguments:
1. NEATM parameter file name
2. lambda (micron)
3. eta
4. pV
Eight arguments:
1. H
2. G
3. alpha (deg)
4. r (AU)
5. delta (AU)
6. lambda (micron)
7. eta
8. pV


In [ ]:
!chmod +x /content/NEATM-master/neatm/c++/neatm

In [ ]:
!make clean

/bin/rm -f main.o neatm.o neatm_input_file.o integral.o  neatm core *~


In [ ]:
cd NEATM-master/


/content/NEATM-master


In [ ]:
!pip install .

Processing /content/NEATM-master
  Preparing metadata (setup.py) ... done
  Created wheel for neatm: filename=neatm-0.7-py3-none-any.whl size=16929 sha256=0dcf1f528ca4ff6dc3b9dcd125713027c40c68da690047a3b581488de600517b
  Stored in directory: /root/.cache/pip/wheels/fd/84/92/a7207846a39bdd0a43e5156eeecd74bda996ebdd10531f8f6e
Successfully built neatm
  Attempting uninstall: neatm
    Found existing installation: neatm 0.7
    Uninstalling neatm-0.7:
      Successfully uninstalled neatm-0.7


In [ ]:
# M.Mueller@astro.rug.nl, 2016/09/05+
# Wrapper around my NEATM executable.
# Output will be in mJy (default) or W/m^2/micron.
# Wavelengths can be an array (iterable) or a scalar Quantity;
# output will be a list of Quantitites or a single Quantity, resp.

from __future__ import print_function
import subprocess
import numpy as np
from astropy import units as u

def neatm(h,g,alpha,r,delta,wavelengths,eta,pv, mJy=True):


    try :
        isIterable = True
        for wav in wavelengths :
            pass
    except TypeError :
        isIterable = False
    if isIterable :
        return [neatm(h,g,alpha,r,delta,wav,eta,pv,mJy=mJy) for wav in wavelengths]
    # wavelengths is scalar:
    try:
        alphaDeg=alpha.to_value(u.degree)
        rAU=r.to_value(u.AU)
        deltaAU=delta.to_value(u.AU)
        lambdaMu=wavelengths.to_value(u.micron)
    except AttributeError:
        print("alpha, r, delta, and lambdaMu must be astropy quantities")
        raise
    except u.UnitConversionError:
        print("Alpha must be an angle; r, delta, and lambdaMu must be (wave-)lengths")
        raise
    cmd=['neatm']
    for var in [h,g,alphaDeg,rAU,deltaAU,lambdaMu,eta,pv]:
        cmd.append(str(var))
    print(cmd)
    process=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    output, cerr = process.communicate()
    output = output.split()
    if mJy:
        return float(output[1])*u.mJy
    else:
        return float(output[0])*u.Watt/u.meter**2/u.micron

h = 1
g = 1
alpha = 1 * u.degree
r = 1 * u.AU
delta = 1 * u.AU
wavelengths = 1* u.micron
eta = 1
pv = 1
mJy = True
neatm(h,g,alpha, r, delta, wavelengths, eta, pv, mJy)

['neatm', '1', '1', '1.0', '1.0', '1.0', '1.0', '1', '1']


<Quantity 2.49152e-28 mJy>